In [22]:
import logging
import os
from tqdm import tqdm
import SimpleITK as sitk
import numpy as np
import sys
from pathlib import Path
from random import randint

log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

MRI_FOLDER = "test_extract"
# ANNOTATION_FOLDER = "output/extract1/labels/"
OUTPUT_DIR = "output/stdtest"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def setup_logger():
    logger = logging.getLogger(__name__)
    
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()
    
    logger.setLevel(logging.DEBUG)
    
    logger.propagate = False
    log_file = os.path.join(log_dir, 'logs_std.log')
    
    # Add file handler
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(logging.DEBUG)
    
    # Add console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO) 
    
    # Create formatter
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    
    # Add handlers
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    
    return logger

# Initialize logger
logger = setup_logger()

NEW_SIZE = [224, 224, 3]

logger.info(f"Starting parameter logging")

logger.debug("Debug logging is enabled")



2025-07-18 16:18:36,705 - INFO - Starting parameter logging


In [23]:
def resize_image_itk(sitk_image, new_size, resample_method=sitk.sitkNearestNeighbor):
    """
    Resize a SimpleITK image to a new size.
    
    Parameters:
    sitk_image (sitk.Image): The input image
    new_size (list or tuple): Target size (should be integers)
    resample_method (int): SimpleITK interpolation method
    
    Returns:
    sitk.Image: Resampled image
    """
    logger.info(f"desired NEW_SIZE is {NEW_SIZE}, received new size is {new_size}")
    new_size = [int(s) for s in new_size]

    original_size = sitk_image.GetSize()
    logger.info(f"original size is {original_size}")
    original_spacing = sitk_image.GetSpacing()

    dim = sitk_image.GetDimension()
    new_spacing = [original_spacing[i] * (original_size[i] / new_size[i]) for i in range(dim)]

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetOutputOrigin(sitk_image.GetOrigin())
    resampler.SetOutputDirection(sitk_image.GetDirection())
    resampler.SetInterpolator(resample_method)
    resampler.SetDefaultPixelValue(0)

    resized_image = resampler.Execute(sitk_image)
    
    return resized_image

In [24]:
if __name__ == "__main__":
    mri_folder = MRI_FOLDER

    output_dir = OUTPUT_DIR
    os.makedirs(output_dir, exist_ok=True)

    mri_paths = [os.path.join(mri_folder, f) for f in os.listdir(mri_folder) 
                if f.endswith('.nii.gz')]
    
    for mri_path in tqdm(mri_paths, desc = "Processing mri_files", unit="file"):
        mri_image = None
        logger.info(f"............Starting process for {mri_path}")
        subj_id = Path(mri_path).stem.split('.')[0]
        try:
            logger.info(f"Loading MRI image from {mri_path}")
            mri_image = sitk.ReadImage(mri_path)
            
            resized_image = resize_image_itk(mri_image, NEW_SIZE)

            logger.info(f"new size is {resized_image.GetSize()}")
            
            output_filename = f"res_{os.path.basename(subj_id)}.nii.gz"
            output_path = os.path.join(OUTPUT_DIR, output_filename)
            logger.info(f"saved file with filename {output_filename}")

            try:
                sitk.WriteImage(resized_image, output_path)
            except Exception as e:
                logger.error(f"error writing image for {mri_path}: {str(e)}")



        except Exception as e:
            logger.error(f"Error processing {mri_path}: {str(e)}")
            continue

Processing mri_files:   0%|          | 0/10 [00:00<?, ?file/s]

2025-07-18 16:18:36,736 - INFO - ............Starting process for test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_1.nii.gz
2025-07-18 16:18:36,737 - INFO - Loading MRI image from test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_1.nii.gz
2025-07-18 16:18:36,744 - INFO - desired NEW_SIZE is [224, 224, 3], received new size is [224, 224, 3]
2025-07-18 16:18:36,745 - INFO - original size is (129, 129, 3)
2025-07-18 16:18:36,750 - INFO - new size is (224, 224, 3)
2025-07-18 16:18:36,751 - INFO - saved file with filename res_image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_1.nii.gz
2025-07-18 16:18:36,788 - INFO - ............Starting process for test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_2.nii.gz
2025-07-18 16:18:36,797 - INFO - Loading MRI image from test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_2.nii.gz
2025-07-18 16:18:36,804 - INFO - desired NEW_SIZE is [224, 224, 3], received new size is [224, 224, 3]
2025-07-18 16:18:36,805 - INFO - original size is (123, 12

Processing mri_files:  20%|██        | 2/10 [00:00<00:00, 18.41file/s]

2025-07-18 16:18:36,846 - INFO - ............Starting process for test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_6.nii.gz
2025-07-18 16:18:36,847 - INFO - Loading MRI image from test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_6.nii.gz
2025-07-18 16:18:36,852 - INFO - desired NEW_SIZE is [224, 224, 3], received new size is [224, 224, 3]
2025-07-18 16:18:36,853 - INFO - original size is (89, 89, 3)
2025-07-18 16:18:36,857 - INFO - new size is (224, 224, 3)
2025-07-18 16:18:36,858 - INFO - saved file with filename res_image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_6.nii.gz
2025-07-18 16:18:36,878 - INFO - ............Starting process for test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_0.nii.gz
2025-07-18 16:18:36,879 - INFO - Loading MRI image from test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_0.nii.gz
2025-07-18 16:18:36,886 - INFO - desired NEW_SIZE is [224, 224, 3], received new size is [224, 224, 3]
2025-07-18 16:18:36,887 - INFO - original size is (131, 131,

Processing mri_files:  50%|█████     | 5/10 [00:00<00:00, 22.61file/s]

2025-07-18 16:18:36,964 - INFO - ............Starting process for test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_4.nii.gz
2025-07-18 16:18:36,965 - INFO - Loading MRI image from test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_4.nii.gz
2025-07-18 16:18:36,970 - INFO - desired NEW_SIZE is [224, 224, 3], received new size is [224, 224, 3]
2025-07-18 16:18:36,971 - INFO - original size is (109, 109, 3)
2025-07-18 16:18:36,976 - INFO - new size is (224, 224, 3)
2025-07-18 16:18:36,977 - INFO - saved file with filename res_image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_4.nii.gz
2025-07-18 16:18:37,001 - INFO - ............Starting process for test_extract/image_856-NPC_T2W_SPIR_TRA+401_node2_2dx3_0.nii.gz
2025-07-18 16:18:37,002 - INFO - Loading MRI image from test_extract/image_856-NPC_T2W_SPIR_TRA+401_node2_2dx3_0.nii.gz
2025-07-18 16:18:37,005 - INFO - desired NEW_SIZE is [224, 224, 3], received new size is [224, 224, 3]
2025-07-18 16:18:37,006 - INFO - original size is (37, 37,

Processing mri_files:  90%|█████████ | 9/10 [00:00<00:00, 27.26file/s]

2025-07-18 16:18:37,088 - INFO - ............Starting process for test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_3.nii.gz
2025-07-18 16:18:37,089 - INFO - Loading MRI image from test_extract/image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_3.nii.gz
2025-07-18 16:18:37,095 - INFO - desired NEW_SIZE is [224, 224, 3], received new size is [224, 224, 3]
2025-07-18 16:18:37,096 - INFO - original size is (117, 117, 3)
2025-07-18 16:18:37,101 - INFO - new size is (224, 224, 3)
2025-07-18 16:18:37,101 - INFO - saved file with filename res_image_856-NPC_T2W_SPIR_TRA+401_node1_2dx3_3.nii.gz


Processing mri_files: 100%|██████████| 10/10 [00:00<00:00, 25.25file/s]
